In [1]:
"""
EE559 Final Project - Handwritten Digit Classification
Dataset: sklearn digits (8x8 proxy; swap fetch_openml for full MNIST)
Models: k-NN, SVM RBF, MLP (PyTorch)
Author: Tung Nguyen | tungdngu@usc.edu
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              classification_report)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

os.makedirs('/content/figs', exist_ok=True)
np.random.seed(42)
torch.manual_seed(42)

# ── 1. LOAD DATASET ──────────────────────────────────────────────────────────
# NOTE: For real MNIST, replace this block with:
#   from sklearn.datasets import fetch_openml
#   mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
#   X, y = mnist.data.astype(np.float32), mnist.target.astype(int)
print("Loading dataset...")
_d = load_digits()
X, y = _d.data.astype(np.float32), _d.target.astype(int)
INPUT_DIM = X.shape[1]
print(f"  Dataset: {X.shape}, classes: {np.unique(y)}")

# 70/10/20 stratified split
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.125, stratify=y_tv, random_state=42)
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ── 2. PREPROCESSING ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

pca_full = PCA(random_state=42)
pca_full.fit(X_train_s)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n90 = np.searchsorted(cumvar, 0.90) + 1
n95 = np.searchsorted(cumvar, 0.95) + 1
n99 = np.searchsorted(cumvar, 0.99) + 1
print(f"  PCA thresholds: 90%={n90}, 95%={n95}, 99%={n99}")

# use 95% variance components for kNN/SVM
pca_red = PCA(n_components=n95, random_state=42)
X_train_pca = pca_red.fit_transform(X_train_s)
X_val_pca   = pca_red.transform(X_val_s)
X_test_pca  = pca_red.transform(X_test_s)


Loading dataset...
  Dataset: (1797, 64), classes: [0 1 2 3 4 5 6 7 8 9]
  Train: (1257, 64), Val: (180, 64), Test: (360, 64)
  PCA thresholds: 90%=30, 95%=39, 99%=53


In [2]:
# ── FIG 1: PCA cumulative variance ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, len(cumvar)+1), cumvar*100, color='#2d4a7a', lw=1.8)
for n, pct, col, label in [(n90,'90%','#e74c3c','90%'),(n95,'95%','#e67e22','95%'),(n99,'99%','#27ae60','99%')]:
    ax.axhline(int(pct[:-1]), color=col, ls='--', lw=1, alpha=0.8)
    ax.axvline(n, color=col, ls='--', lw=1, alpha=0.8, label=f'{label}: {n} comps')
ax.set_xlabel('Number of Components', fontsize=10)
ax.set_ylabel('Cumulative Explained Variance (%)', fontsize=10)
ax.set_title('PCA Cumulative Explained Variance', fontsize=11, fontweight='bold')
ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/figs/fig_pca_variance.pdf', bbox_inches='tight')
plt.close(); print("  Fig 1 saved.")

# ── FIG 2: sample digit images ───────────────────────────────────────────────
side = int(np.sqrt(INPUT_DIM))
fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(X_train[idx].reshape(side, side), cmap='gray_r')
    axes[0, digit].axis('off'); axes[0, digit].set_title(str(digit), fontsize=9)
    axes[1, digit].imshow(X_train_s[idx].reshape(side, side), cmap='RdBu_r')
    axes[1, digit].axis('off')
axes[0,0].set_ylabel('Raw', fontsize=8, rotation=0, labelpad=22)
axes[1,0].set_ylabel('Scaled', fontsize=8, rotation=0, labelpad=18)
plt.suptitle('Sample Digits: Raw vs. Normalized', fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('/content/figs/fig_samples.pdf', bbox_inches='tight')
plt.close(); print("  Fig 2 saved.")


  Fig 1 saved.
  Fig 2 saved.


In [3]:

# ── 3. k-NN ──────────────────────────────────────────────────────────────────
print("\nTuning k-NN...")
k_vals = [1, 3, 5, 7, 9, 11, 15]
knn_val_accs = []
for k in k_vals:
    m = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    m.fit(X_train_pca, y_train)
    acc = accuracy_score(y_val, m.predict(X_val_pca))
    knn_val_accs.append(acc)
    print(f"  k={k}: val={acc:.4f}")
best_k = k_vals[int(np.argmax(knn_val_accs))]

knn_best = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', n_jobs=-1)
knn_best.fit(X_train_pca, y_train)
knn_test_pred = knn_best.predict(X_test_pca)
knn_test_acc  = accuracy_score(y_test, knn_test_pred)
knn_test_f1   = f1_score(y_test, knn_test_pred, average='weighted')
knn_cm = confusion_matrix(y_test, knn_test_pred)
print(f"  Best k={best_k} -> Test acc={knn_test_acc:.4f}, F1={knn_test_f1:.4f}")

# ── FIG 3: kNN tuning ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(k_vals, [a*100 for a in knn_val_accs], 'o-', color='#2d4a7a', ms=7, lw=2)
ax.axvline(best_k, color='#e74c3c', ls='--', lw=1.5, label=f'Best k={best_k}')
ax.set_xlabel('k (Neighbors)', fontsize=10)
ax.set_ylabel('Validation Accuracy (%)', fontsize=10)
ax.set_title('k-NN Hyperparameter Tuning', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
y_min = min(knn_val_accs)*100
ax.set_ylim(max(0, y_min - 2), 101)
plt.tight_layout()
plt.savefig('/content/figs/fig_knn_tuning.pdf', bbox_inches='tight')
plt.close(); print("  Fig 3 saved.")


Tuning k-NN...
  k=1: val=0.9722
  k=3: val=0.9667
  k=5: val=0.9722
  k=7: val=0.9778
  k=9: val=0.9667
  k=11: val=0.9722
  k=15: val=0.9611
  Best k=7 -> Test acc=0.9611, F1=0.9609
  Fig 3 saved.


In [4]:

# ── 4. SVM ───────────────────────────────────────────────────────────────────
print("\nTuning SVM...")
C_vals = [0.1, 1, 10, 50]
svm_val_accs = []
for C in C_vals:
    m = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    m.fit(X_train_pca, y_train)
    acc = accuracy_score(y_val, m.predict(X_val_pca))
    svm_val_accs.append(acc)
    print(f"  C={C}: val={acc:.4f}")
best_C = C_vals[int(np.argmax(svm_val_accs))]

svm_best = SVC(kernel='rbf', C=best_C, gamma='scale', random_state=42)
svm_best.fit(X_train_pca, y_train)
svm_test_pred = svm_best.predict(X_test_pca)
svm_test_acc  = accuracy_score(y_test, svm_test_pred)
svm_test_f1   = f1_score(y_test, svm_test_pred, average='weighted')
svm_cm = confusion_matrix(y_test, svm_test_pred)
print(f"  Best C={best_C} -> Test acc={svm_test_acc:.4f}, F1={svm_test_f1:.4f}")
# ── FIG 4: SVM tuning ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(range(len(C_vals)), [a*100 for a in svm_val_accs], 's-', color='#8e44ad', ms=7, lw=2)
ax.axvline(C_vals.index(best_C), color='#e74c3c', ls='--', lw=1.5, label=f'Best C={best_C}')
ax.set_xticks(range(len(C_vals))); ax.set_xticklabels([str(c) for c in C_vals])
ax.set_xlabel('C (Regularization)', fontsize=10)
ax.set_ylabel('Validation Accuracy (%)', fontsize=10)
ax.set_title('SVM RBF Hyperparameter Tuning', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
y_min2 = min(svm_val_accs)*100
ax.set_ylim(max(0, y_min2 - 1), 101)
plt.tight_layout()
plt.savefig('/content/figs/fig_svm_tuning.pdf', bbox_inches='tight')
plt.close(); print("  Fig 4 saved.")



Tuning SVM...
  C=0.1: val=0.9389
  C=1: val=0.9833
  C=10: val=0.9778
  C=50: val=0.9778
  Best C=1 -> Test acc=0.9778, F1=0.9777
  Fig 4 saved.


In [5]:
# ── 5. MLP ───────────────────────────────────────────────────────────────────
print("\nTraining MLP...")
device = torch.device('cpu')

class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        # architecture scales with input dim:
        # digits(64): 64->128->64->32->10
        # MNIST(784): 784->512->256->128->10
        h1 = min(512, max(128, in_dim * 4))
        h2 = h1 // 2; h3 = h2 // 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h1, h2),    nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h2, h3),    nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(h3, 10)
        )
    def forward(self, x): return self.net(x)

Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train,   dtype=torch.long)
Xvl = torch.tensor(X_val_s,   dtype=torch.float32)
yvl = torch.tensor(y_val,     dtype=torch.long)
Xts = torch.tensor(X_test_s,  dtype=torch.float32)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

model = MLP(INPUT_DIM).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_acc, patience_count, PATIENCE = 0, 0, 10

for epoch in range(1, 61):
    model.train()
    ep_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        ep_loss += loss.item() * len(xb)
    scheduler.step()

    model.eval()
    with torch.no_grad():
        vl_out  = model(Xvl)
        vl_loss = criterion(vl_out, yvl).item()
        tr_pred = model(Xtr).argmax(1).numpy()
        vl_pred = vl_out.argmax(1).numpy()

    tr_acc = accuracy_score(y_train, tr_pred)
    vl_acc = accuracy_score(y_val,   vl_pred)
    train_losses.append(ep_loss / len(Xtr))
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/content/best_mlp.pt')
        patience_count = 0
    else:
        patience_count += 1

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:02d}: loss={train_losses[-1]:.4f}, val_acc={vl_acc:.4f}")
    if patience_count >= PATIENCE:
        print(f"  Early stopping at epoch {epoch}"); break

model.load_state_dict(torch.load('/content/best_mlp.pt', weights_only=True))
model.eval()
with torch.no_grad():
    mlp_test_pred = model(Xts).argmax(1).numpy()
mlp_test_acc = accuracy_score(y_test, mlp_test_pred)
mlp_test_f1  = f1_score(y_test, mlp_test_pred, average='weighted')
mlp_cm = confusion_matrix(y_test, mlp_test_pred)
print(f"  MLP Test acc={mlp_test_acc:.4f}, F1={mlp_test_f1:.4f}")



Training MLP...
  Epoch 10: loss=0.1615, val_acc=0.9889
  Epoch 20: loss=0.0701, val_acc=0.9889
  Early stopping at epoch 20
  MLP Test acc=0.9833, F1=0.9834


In [6]:
# ── FIG 5: MLP learning curves ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ep = range(1, len(train_losses)+1)
ax1.plot(ep, train_losses, label='Train', color='#2d4a7a', lw=1.8)
ax1.plot(ep, val_losses,   label='Val',   color='#e74c3c', ls='--', lw=1.8)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('MLP Loss Curves', fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, [a*100 for a in train_accs], label='Train Acc', color='#2d4a7a', lw=1.8)
ax2.plot(ep, [a*100 for a in val_accs],   label='Val Acc',   color='#e74c3c', ls='--', lw=1.8)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('MLP Accuracy Curves', fontweight='bold'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/figs/fig_mlp_curves.pdf', bbox_inches='tight')
plt.close(); print("  Fig 5 saved.")

# ── FIG 6: Confusion matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, cm, title, cmap in zip(axes,
        [knn_cm, svm_cm, mlp_cm],
        [f'k-NN (k={best_k})', f'SVM RBF (C={best_C})', 'MLP'],
        ['Blues', 'Purples', 'Greens']):
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_n, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(range(10), fontsize=8); ax.set_yticklabels(range(10), fontsize=8)
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle('Normalized Confusion Matrices', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/figs/fig_confusion.pdf', bbox_inches='tight')
plt.close(); print("  Fig 6 saved.")

# ── FIG 7: final comparison bar chart ────────────────────────────────────────
models_label = [f'k-NN\n(k={best_k})', f'SVM RBF\n(C={best_C})', 'MLP\n(4-layer)']
accs = [knn_test_acc*100, svm_test_acc*100, mlp_test_acc*100]
f1s  = [knn_test_f1*100,  svm_test_f1*100,  mlp_test_f1*100]
x = np.arange(3)
fig, ax = plt.subplots(figsize=(6, 4))
b1 = ax.bar(x-0.2, accs, 0.35, label='Accuracy (%)', color='#2d4a7a', alpha=0.85)
b2 = ax.bar(x+0.2, f1s,  0.35, label='F1-Score (%)', color='#e67e22', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models_label, fontsize=9)
ax.set_ylabel('Score (%)'); ax.set_title('Final Test Performance Comparison', fontweight='bold')
ax.set_ylim(max(0, min(accs+f1s)-5), 103)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
for bar in list(b1)+list(b2):
    ax.annotate(f'{bar.get_height():.2f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=7.5)
plt.tight_layout()
plt.savefig('/content/figs/fig_comparison.pdf', bbox_inches='tight')
plt.close(); print("  Fig 7 saved.")


  Fig 5 saved.
  Fig 6 saved.
  Fig 7 saved.


In [7]:

# ── SAVE RESULTS ─────────────────────────────────────────────────────────────
results = dict(
    knn_test_acc=knn_test_acc, knn_test_f1=knn_test_f1, best_k=best_k,
    knn_val_accs=knn_val_accs, k_vals=k_vals,
    svm_test_acc=svm_test_acc, svm_test_f1=svm_test_f1, best_C=best_C,
    svm_val_accs=svm_val_accs, C_vals=C_vals,
    mlp_test_acc=mlp_test_acc, mlp_test_f1=mlp_test_f1,
    train_losses=train_losses, val_losses=val_losses,
    train_accs=train_accs, val_accs=val_accs,
    n90=n90, n95=n95, n99=n99,
    n_train=len(X_train), n_val=len(X_val), n_test=len(X_test),
    INPUT_DIM=INPUT_DIM,
)
np.save('/content/final_results.npy', results)
print("\n=== FINAL RESULTS ===")
print(f"k-NN  (k={best_k}):  acc={knn_test_acc:.4f}, F1={knn_test_f1:.4f}")
print(f"SVM   (C={best_C}):  acc={svm_test_acc:.4f}, F1={svm_test_f1:.4f}")
print(f"MLP:                  acc={mlp_test_acc:.4f}, F1={mlp_test_f1:.4f}")
print("Done!")


=== FINAL RESULTS ===
k-NN  (k=7):  acc=0.9611, F1=0.9609
SVM   (C=1):  acc=0.9778, F1=0.9777
MLP:                  acc=0.9833, F1=0.9834
Done!
